In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

from datasets import load_dataset

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")

In [ ]:
ROOT_PATH = Path("../").resolve()
DATA_PATH = ROOT_PATH / "data"

In [ ]:
# repo_id = "stanfordnlp/sst2"  # Other: m-a-p/FineFineWeb-sample, google/civil_comments, toxigen/toxigen-data, SetFit/bbc-news
# local_dir = "../data/sst2"

# downloaded_path = snapshot_download(
#     repo_id=repo_id,
#     repo_type="dataset",
#     local_dir=local_dir,
# )


repo_id = "8Opt/bert-mlm-experiments-en"
split = "test[:100]"
ds = load_dataset(repo_id, split=split)
ds

In [ ]:
ds.num_rows

In [ ]:
import torch
from transformers import BertTokenizerFast


class BertTokenizer:
    def __init__(self, model_id: str = "bert-base-uncased"):
        self.tokenizer = BertTokenizerFast.from_pretrained(model_id)

    def tokenize(self, text: str) -> list[str]:
        return self.tokenizer.tokenize(text)

    def encode(
        self,
        text: str | list[str],
        padding: str | bool = "max_length",
        truncation: bool = True,
        max_length: int | None = 512,
        return_tensors: str | None = "pt",
    ) -> dict[str, torch.Tensor]:
        """Encodes a string or list of strings into token IDs and attention masks."""
        return self.tokenizer(
            text,
            padding=padding,
            truncation=truncation,
            max_length=max_length,
            return_tensors=return_tensors,
            return_attention_mask=True,
        )

    def decode(self, token_ids: list[int], skip_special_tokens: bool = True) -> str:
        return self.tokenizer.decode(token_ids, skip_special_tokens=skip_special_tokens)


# Instantiate the class cleanly
tokenizer = BertTokenizer()

In [ ]:
sample = ds[0]
sample

In [ ]:
tokenizer.encode(text=sample["text"]).keys()

In [ ]:
tokenized_ds = []

for data in ds:
    _tok_data = tokenizer.encode(data["text"])
    tokenized_ds.append(_tok_data)

In [ ]:
import torch
from torch.utils.data import Dataset
import datasets


class LfbDataset(Dataset):
    """Supports both MLM pretraining and downstream task fine-tuning."""

    def __init__(
        self,
        dataset: datasets.Dataset,
        tokenizer: BertTokenizer,
        max_length: int = 512,
        text_column: str = "text",
        label_column: str | None = None,
        mlm: bool = True,
        mlm_probability: float = 0.15,
    ):

        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.text_column = text_column
        self.label_column = label_column
        self.mlm = mlm
        self.mlm_probability = mlm_probability

        # Validate columns
        if text_column not in dataset.column_names:
            raise ValueError(f"Column '{text_column}' not found")
        if label_column and label_column not in dataset.column_names:
            raise ValueError(f"Column '{label_column}' not found")

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx: int) -> dict:
        item = self.dataset[idx]

        # Tokenize
        encoded = self.tokenizer.encode(
            item[self.text_column],
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors=None,
        )

        output = {
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
        }

        # MLM: mask tokens for pretraining
        if self.mlm:
            output["labels"] = self._create_mlm_labels(output["input_ids"].copy())

        # Downstream: add task labels
        if self.label_column:
            output["labels"] = item[self.label_column]

        return output

    def _create_mlm_labels(self, input_ids: list[int]) -> list[int]:
        """Create MLM labels."""
        labels = [-100] * len(input_ids)
        tokenizer_obj = self.tokenizer.tokenizer

        for idx, token_id in enumerate(input_ids):
            # Skip special tokens
            if token_id in [
                tokenizer_obj.cls_token_id,
                tokenizer_obj.sep_token_id,
                tokenizer_obj.pad_token_id,
            ]:
                continue

            # Mask randomly
            if torch.rand(1).item() < self.mlm_probability:
                input_ids[idx] = tokenizer_obj.mask_token_id
                labels[idx] = token_id

        return labels

In [ ]:
test_ds = LfbDataset(dataset=ds, tokenizer=tokenizer, mlm_probability=0.3)

In [ ]:
from typing import List, Dict


def collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    """Convert batch dicts to stacked tensors."""
    output = {}

    for key in batch[0].keys():
        values = [item[key] for item in batch]

        # Convert to tensor
        if isinstance(values[0], list):
            output[key] = torch.tensor(values, dtype=torch.long)
        else:
            output[key] = torch.stack(values)

    return output

In [ ]:
# collate_fn(lfb_samples)